In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

Matplotlib is building the font cache; this may take a moment.


In [92]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
import shap

In [1]:
from xgboost import XGBClassifier

In [4]:
#Load data
df = pd.read_csv("../data/mental_health_data.csv")
df.head()

,Name,Age,Marital_Status,Education_Level,Number_of_Children,Smoking_Status,Physical_Activity_Level,Employment_Status,Income,Alcohol_Consumption,Dietary_Habits,Sleep_Patterns,History_of_Mental_Illness,History_of_Substance_Abuse,Family_History_of_Depression,Chronic_Medical_Conditions,Record_ID,data_split
0,Christine Barker,31,Married,Bachelor's Degree,2,Non-smoker,Active,Unemployed,26265.67,Moderate,Moderate,Fair,Yes,No,Yes,Yes,100000,validation
1,Jacqueline Lewis,55,Married,High School,1,Non-smoker,Sedentary,Employed,42710.36,High,Unhealthy,Fair,Yes,No,No,Yes,100001,validation
2,Shannon Church,78,Widowed,Master's Degree,1,Non-smoker,Sedentary,Employed,125332.79,Low,Unhealthy,Good,No,No,Yes,No,100002,validation
3,Charles Jordan,58,Divorced,Master's Degree,3,Non-smoker,Moderate,Unemployed,9992.78,Moderate,Moderate,Poor,No,No,No,No,100003,validation
4,Michael Rich,18,Single,High School,0,Non-smoker,Sedentary,Unemployed,8595.08,Low,Moderate,Fair,Yes,No,Yes,Yes,100004,validation


From the EDA, we had a general understanding  of the data, including the distribution of features, the relationship between features and the target variable. In this scripts, I'll start testing the model and some hypeterparameter tuning. 

## Transformation

#### Categorical features

In [5]:
exl_cols = ["Name", "Record_ID", "data_split", "History_of_Mental_Illness"]

In [6]:
df["Number_of_Children"] = df["Number_of_Children"].replace(4, 3)

I would not group widowed and divorced together in this case. Normally, considering the different use cases and how the results will be used, I would apply some consideration to treating vulnerable customers. If the results could result in treating customers unfairly, then I would group vulnerable customers into another set. However, in this specific situation, I will not take any action, as there is an important link between marital status and the likelihood of depression.

I will treat PHD as a distinct education level, as merging it with Master's would remove information from the model. If education is associated with differences in the target, combining PhD with Master's could hide that difference.

In [ ]:
train = df[df["data_split"] == "train"].copy()
validation = df[df["data_split"] == "validation"].copy()
test = df[df["data_split"] == "test"].copy()

X_train = train.drop(columns=exl_cols)
X_val = validation.drop(columns=exl_cols)
X_test = test.drop(columns=exl_cols)


In [62]:
y_train = train["History_of_Mental_Illness"]
y_val = validation["History_of_Mental_Illness"]
y_test = test["History_of_Mental_Illness"]

In [8]:
numeric_features = ["Age", "Income"]
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()
categorical_features.append("Number_of_Children")
categorical_features

['Marital_Status',
 'Education_Level',
 'Smoking_Status',
 'Physical_Activity_Level',
 'Employment_Status',
 'Alcohol_Consumption',
 'Dietary_Habits',
 'Sleep_Patterns',
 'History_of_Substance_Abuse',
 'Family_History_of_Depression',
 'Chronic_Medical_Conditions',
 'Number_of_Children']

In [9]:
#Frequent encoding for categorical features
marital_frequency = (X_train["Marital_Status"].value_counts(normalize=True))
X_train["Marital_Status"] = (X_train["Marital_Status"].map(marital_frequency).fillna(0))
X_val["Marital_Status"] = (X_val["Marital_Status"].map(marital_frequency).fillna(0))
X_test["Marital_Status"] = (X_test["Marital_Status"].map(marital_frequency).fillna(0))

In [65]:
#Orindial encoding 
ordinal_features = ["Education_Level", "Physical_Activity_Level", "Sleep_Patterns","Alcohol_Consumption","Dietary_Habits", "Number_of_Children", "Smoking_Status"]

In [66]:
ordinal_categories = [
    [   "High School",
        "Associate Degree",
        "Bachelor's Degree",
        "Master's Degree",
        "PhD"
    ],

    [   "Sedentary",
        "Moderate",
        "Active"
    ],

    [   "Poor",
        "Fair",
        "Good"
    ],

    [   "Low",
        "Moderate",
        "High"
    ],

    [
        "Unhealthy",
        "Moderate",
        "Healthy"
    ],

    [
        "0",
        "1",
        "2",
        "3"
    ],

    [
        "Non-Smoker",
        "Former",
        "Current"
    ]
]

In [67]:
ordinal_encoder = OrdinalEncoder(
    categories=ordinal_categories,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

In [53]:
# Convert True/False to 0/1
binary_features = [
    "Employment_Status",
    "History_of_Substance_Abuse",
    "Family_History_of_Depression",
    "Chronic_Medical_Conditions"
]

binary_mappings = {
    "Yes": 1,
    "No": 0,
    "Employed": 1,
    "Unemployed": 0
}

for feature in binary_features:
    X_train[feature] = X_train[feature].map(binary_mappings)
    X_val[feature] = X_val[feature].map(binary_mappings)
    X_test[feature] = X_test[feature].map(binary_mappings)

# Employment status
for data in [X_train, X_val, X_test]:
    data["Employment_Status"] = data["Employment_Status"].map({
        "Employed": 1,
        "Unemployed": 0
    })

#### Numerical Feataures 

In [15]:
#Numeric imputer
numeric_features

['Age', 'Income']

From EDA, we noticed that Income is a right-skewed feature, so normally we could take some mathmatical transformation to it, like Log(x) to reduce the effect of the long right tail.  However, using the original Income values makes the model output much easier explain to stakeholders than log(x).Particularly when using a more complex model such as XGBoost. Therefore, I'd like retain Income in its original scale and apply a cap to limit the influence of extreme values.

In [16]:
df["Income"] = df["Income"].clip(upper=150000)
X_train["Income"] = X_train["Income"].clip(upper=150000)
X_val["Income"] = X_val["Income"].clip(upper=150000)
X_test["Income"] = X_test["Income"].clip(upper=150000)

In [75]:
new_num_features = [
    "Age",
    "Income",
    "Marital_Status",
    "Employment_Status",
    "History_of_Substance_Abuse",
    "Family_History_of_Depression",
    "Chronic_Medical_Conditions"
]

In [76]:
preprocessor = ColumnTransformer(
    transformers=[
        ("ordinal", ordinal_encoder, ordinal_features),
        ("numeric", "passthrough", new_num_features),
    ]
)

In [77]:
X_train_transformed = preprocessor.fit_transform(X_train)

In [78]:
X_val_transformed = preprocessor.transform(X_val)
X_test_transformed = preprocessor.transform(X_test)

## Train model

In [63]:
y_train = y_train.map({"No": 0, "Yes": 1})
y_val = y_val.map({"No": 0, "Yes": 1})
y_test = y_test.map({"No": 0, "Yes": 1})

In [79]:
## Train a default XGboost Model 
xgb_model = XGBClassifier(
    objective = "binary:logistic",
    learning_rate = 0.3,
    max_depth = 5,
    min_child_weight =1,
    subsample =0.8,
    gamma = 0,
    reg_alpha=1,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    enable_categorical=True
)

xgb_model.fit(
    X_train_transformed,
    y_train,
    eval_set=[(X_val_transformed, y_val)],
)

[0]	validation_0-logloss:0.60526
[1]	validation_0-logloss:0.60182
[2]	validation_0-logloss:0.60014
[3]	validation_0-logloss:0.59934
[4]	validation_0-logloss:0.59896
[5]	validation_0-logloss:0.59872
[6]	validation_0-logloss:0.59868
[7]	validation_0-logloss:0.59866
[8]	validation_0-logloss:0.59866
[9]	validation_0-logloss:0.59875
[10]	validation_0-logloss:0.59875
[11]	validation_0-logloss:0.59875
[12]	validation_0-logloss:0.59874
[13]	validation_0-logloss:0.59874
[14]	validation_0-logloss:0.59877
[15]	validation_0-logloss:0.59880
[16]	validation_0-logloss:0.59882
[17]	validation_0-logloss:0.59883
[18]	validation_0-logloss:0.59885
[19]	validation_0-logloss:0.59891
[20]	validation_0-logloss:0.59896
[21]	validation_0-logloss:0.59902
[22]	validation_0-logloss:0.59905
[23]	validation_0-logloss:0.59909
[24]	validation_0-logloss:0.59912
[25]	validation_0-logloss:0.59910
[26]	validation_0-logloss:0.59910
[27]	validation_0-logloss:0.59915
[28]	validation_0-logloss:0.59915
[29]	validation_0-loglos

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,50
,enable_categorical,True
,eval_metric,None


In [88]:
#Gain importance
feature_list = preprocessor.get_feature_names_out()
booster = xgb_model.get_booster()
gain_importance = booster.get_score(importance_type="gain")
gain_values = [
    gain_importance.get(f"f{i}", 0)
    for i in range(len(feature_list))
]

In [91]:
#SHAP
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_val_transformed)
shap_importance = np.abs(shap_values).mean(axis=0)

In [95]:
perm_result = permutation_importance(
    xgb_model,
    X_val_transformed,
    y_val,
    scoring="roc_auc"
)

permutation_values = perm_result.importances_mean

In [97]:
feature_importance = pd.DataFrame({
    "Feature": feature_list,
    "Gain": gain_values,
    "SHAP": shap_importance,
    "Permutation": permutation_values
})

feature_importance["Gain_Rank"] = feature_importance["Gain"].rank(ascending=False)
feature_importance["SHAP_Rank"] = feature_importance["SHAP"].rank(ascending=False)
feature_importance["Permutation_Rank"] = feature_importance["Permutation"].rank(ascending=False)

feature_importance["Overall_Rank"] = (
    feature_importance[
        ["Gain_Rank", "SHAP_Rank", "Permutation_Rank"]
    ].mean(axis=1)
)

feature_importance = feature_importance.sort_values("Overall_Rank", ascending=True)
feature_importance

,Feature,Gain,SHAP,Permutation,Gain_Rank,SHAP_Rank,Permutation_Rank,Overall_Rank
8,numeric__Income,19.080156,0.289318,0.092314,1.0,1.0,1.0,1.000000
0,ordinal__Education_Level,9.842602,0.050093,0.017042,3.0,2.0,2.0,2.333333
6,ordinal__Smoking_Status,15.824492,0.034437,0.009858,2.0,3.0,3.0,2.666667
7,numeric__Age,3.307952,0.010381,0.000280,4.0,4.0,5.0,4.333333
3,ordinal__Alcohol_Consumption,3.272462,0.004408,0.000216,6.0,7.0,6.0,6.333333
1,ordinal__Physical_Activity_Level,3.296414,0.002954,0.000155,5.0,8.0,7.0,6.666667
4,ordinal__Dietary_Habits,3.018100,0.002841,0.000306,8.0,9.0,4.0,7.000000
2,ordinal__Sleep_Patterns,2.974053,0.005503,0.000142,10.0,6.0,8.0,8.000000
9,numeric__Marital_Status,3.008533,0.005558,-0.000045,9.0,5.0,13.0,9.000000
5,ordinal__Number_of_Children,3.076799,0.002250,-0.000109,7.0,10.0,14.0,10.333333


based on the results, ill remove bottom features.